# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.items import Item
from litellm import completion
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [8]:
LITE_MODE = True

load_dotenv(override=True)

HF_TOKEN = os.environ['HF_TOKEN']

login(token=HF_TOKEN, add_to_git_credential=True)


# Load the data



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
user_name ='ed-donner'
dataset_name = f"{user_name}/items_lite" if LITE_MODE else f"{user_name}/items_full"

train,test,split = Item.from_hub(dataset_name)
print(f"Loaded {len(train)} training and {len(test)} test items")

# Create a smaller subset for testing



Loaded 20000 training and 1000 test items


## you can try it for human neural network

# Lets try vanilla neural n/w

In [10]:
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]


In [11]:
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [12]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [13]:
# convert the data to sensors

X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.1, random_state=42)

#create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)


In [14]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [17]:
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 16762.389, Val Loss: 19370.693


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 8070.389, Val Loss: 18196.820


In [19]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)
    

In [20]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$34 $56 $103 $13 $5 $152 $61 $24 $14 $25 $81 $71 $23 $16 $95 $86 $8 $18 $16 $66 $85 $149 $34 $29 $686 $182 $212 $57 $187 $93 $14 $68 $45 $34 $6 $24 $26 $48 $634 $57 $107 $25 $50 $14 $43 $33 $337 $68 $78 $189 $14 $145 $45 $31 $28 $2 $53 $114 $21 $125 $31 $25 $24 $48 $222 $87 $18 $32 $20 $12 $119 $153 $67 $22 $23 $65 $17 $175 $45 $123 $40 $47 $38 $54 $85 $17 $67 $24 $129 $16 $74 $82 $55 $50 $50 $497 $1 $22 $69 $443 $165 $13 $47 $76 $5 $104 $167 $479 $64 $441 $44 $207 $88 $15 $52 $125 $31 $2 $26 $33 $19 $20 $61 $24 $202 $26 $89 $591 $35 $38 $31 $104 $61 $40 $61 $24 $94 $71 $27 $165 $52 $100 $63 $52 $116 $83 $12 $29 $28 $45 $6 $35 $41 $22 $19 $415 $27 $17 $93 $77 $19 $126 $32 $2 $75 $42 $11 $147 $62 $29 $41 $25 $31 $113 $32 $35 $155 $174 $34 $58 $43 $37 $92 $54 $15 $29 $56 $30 $81 $3 $36 $62 $55 $15 $314 $116 $96 $144 $295 $132 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [21]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [23]:
print(messages_for(test[0]))

[{'role': 'user', 'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: SAFUEL 10,000mAh Magnetic Portable Charger  \nCategory: Electronics  \nBrand: SAFUEL  \nDescription: A compact 10,000mAh power bank that snaps magnetically to MagSafe‑enabled iPhones for fast wireless charging.  \nDetails: Offers 20W USB‑C PD, QC 3.0 wired output, auto shut‑off, and a built‑in LED indicator.'}]


In [24]:
def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [25]:
evaluate(gpt_4__1_nano,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$12 $112 $25 $15 $61 $89 $11 $28 $39 $70 $150 $39 $36 $24 $2 $2 $120 $5 $12 $15 $10 $50 $0 $4 $51 $75 $245 $21 $56 $25 $8 $9 $10 $10 $58 $27 $36 $133 $150 $1 $12 $7 $0 $14 $5 $22 $23 $12 $34 $120 $31 $25 $35 $81 $6 $108 $900 $70 $42 $14 $4 $6 $9 $75 $192 $20 $75 $6 $48 $99 $20 $31 $1 $9 $79 $0 $156 $246 $8 $14 $66 $19 $5 $230 $80 $50 $53 $25 $2 $90 $50 $34 $49 $60 $28 $1371 $5 $43 $16 $250 $198 $20 $18 $16 $53 $7 $52 $300 $55 $281 $3 $6 $254 $90 $99 $21 $4 $21 $62 $141 $70 $23 $65 $9 $125 $10 $19 $300 $155 $2 $0 $189 $29 $134 $41 $62 $0 $50 $52 $129 $84 $39 $19 $54 $3 $101 $1 $50 $0 $1 $4 $30 $130 $0 $39 $420 $0 $16 $145 $50 $50 $87 $5 $64 $4 $3 $90 $901 $65 $3 $13 $25 $50 $92 $6 $9 $155 $120 $50 $34 $4 $65 $197 $21 $0 $1 $10 $59 $20 $66 $11 $220 $7 $225 $73 $70 $60 $0 $150 $100 

In [ ]:

import xgboost as xgb

In [26]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [27]:
evaluate(gemini_2__5_flash_lite,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$2 $12 $65 $5 $41 $89 $11 $18 $38 $320 
Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

$125 
Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://git

RateLimitError: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash-lite\nPlease retry in 2.746618792s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "gene